In [1]:
# LGBM
# sklearn environment
# uses pandas, numpy, sklearn, and lightgbm

# The Data

In [2]:
import pandas as pd
import numpy as np

In [3]:
train_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')

In [4]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
test_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [6]:
y_train = train_df['SalePrice']
train_df.drop(['SalePrice'], inplace=True, axis=1)

In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 80 columns):
Id               1460 non-null int64
MSSubClass       1460 non-null int64
MSZoning         1460 non-null object
LotFrontage      1201 non-null float64
LotArea          1460 non-null int64
Street           1460 non-null object
Alley            91 non-null object
LotShape         1460 non-null object
LandContour      1460 non-null object
Utilities        1460 non-null object
LotConfig        1460 non-null object
LandSlope        1460 non-null object
Neighborhood     1460 non-null object
Condition1       1460 non-null object
Condition2       1460 non-null object
BldgType         1460 non-null object
HouseStyle       1460 non-null object
OverallQual      1460 non-null int64
OverallCond      1460 non-null int64
YearBuilt        1460 non-null int64
YearRemodAdd     1460 non-null int64
RoofStyle        1460 non-null object
RoofMatl         1460 non-null object
Exterior1st      1460 non-n

In [8]:
drop = ['Id', 'Alley', 'PoolQC', 'MiscFeature', 'Fence', 'FireplaceQu']
train_df.drop(drop, axis=1, inplace=True)
test_df.drop(drop, axis=1, inplace=True)

# Impute

In [9]:
# Can just run SimpleImputer over all of the columns;
#   with a separate one for numeric columns and one for categorical columns

# Import
from sklearn.impute import SimpleImputer

In [10]:
# Imputer for numeric columns
mean_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

# Imputer for categorical columns
common_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')

In [11]:
# found this function; really like it
numerical = train_df.select_dtypes(exclude=['object']).columns
categorical = train_df.select_dtypes(exclude=['float64', 'int64']).columns

In [12]:
# fit the imputers
mean_imputer.fit(train_df[numerical])
common_imputer.fit(train_df[categorical])

SimpleImputer(add_indicator=False, copy=True, fill_value=None,
              missing_values=nan, strategy='most_frequent', verbose=0)

In [13]:
# transform the data - numeric
train_df[numerical] = mean_imputer.transform(train_df[numerical]);
test_df[numerical] = mean_imputer.transform(test_df[numerical]);

# transform the data - categorical
train_df[categorical] = common_imputer.transform(train_df[categorical]);
test_df[categorical] = common_imputer.transform(test_df[categorical]);

In [14]:
train_df.columns[train_df.isnull().any()]

Index([], dtype='object')

In [15]:
test_df.columns[test_df.isnull().any()]

Index([], dtype='object')

In [16]:
# Quick correlation with SalePrice
pd.concat([train_df, y_train], axis=1).corr()['SalePrice'].sort_values(ascending=False)

SalePrice        1.000000
OverallQual      0.790982
GrLivArea        0.708624
GarageCars       0.640409
GarageArea       0.623431
TotalBsmtSF      0.613581
1stFlrSF         0.605852
FullBath         0.560664
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
YearRemodAdd     0.507101
MasVnrArea       0.475241
GarageYrBlt      0.470177
Fireplaces       0.466929
BsmtFinSF1       0.386420
LotFrontage      0.334901
WoodDeckSF       0.324413
2ndFlrSF         0.319334
OpenPorchSF      0.315856
HalfBath         0.284108
LotArea          0.263843
BsmtFullBath     0.227122
BsmtUnfSF        0.214479
BedroomAbvGr     0.168213
ScreenPorch      0.111447
PoolArea         0.092404
MoSold           0.046432
3SsnPorch        0.044584
BsmtFinSF2      -0.011378
BsmtHalfBath    -0.016844
MiscVal         -0.021190
LowQualFinSF    -0.025606
YrSold          -0.028923
OverallCond     -0.077856
MSSubClass      -0.084284
EnclosedPorch   -0.128578
KitchenAbvGr    -0.135907
Name: SalePrice, dtype: float64

# Encode

In [17]:
# These need encoding
categorical.to_numpy()

array(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1',
       'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', 'KitchenQual', 'Functional',
       'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
       'PavedDrive', 'SaleType', 'SaleCondition'], dtype=object)

In [18]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse=False, categories='auto')

In [19]:
# Get the categorical columns
train_cat = train_df[categorical]
test_cat = test_df[categorical]

In [20]:
# Apply the OHE
train_ohe = ohe.fit_transform(train_cat)
test_ohe = ohe.transform(test_cat)

In [21]:
test_ohe

array([[0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [22]:
# Drop the old columns
train_df.drop(categorical, axis=1, inplace=True)
test_df.drop(categorical, axis=1, inplace=True)

In [23]:
# Concatenate what is left with the ohe columns
train_ready = pd.concat([train_df, pd.DataFrame(train_ohe)], axis=1)
test_ready = pd.concat([test_df, pd.DataFrame(test_ohe)], axis=1)

In [24]:
# Quick correlation with SalePrice
pd.concat([train_ready, y_train], axis=1).corr()['SalePrice'].sort_values(ascending=False)

# some ohe categories are fairly strongly negatively correlated

SalePrice      1.000000
OverallQual    0.790982
GrLivArea      0.708624
GarageCars     0.640409
GarageArea     0.623431
                 ...   
127           -0.367456
147           -0.498545
205           -0.513906
189           -0.519298
132           -0.589044
Name: SalePrice, Length: 271, dtype: float64

# Model

In [25]:
# Grid search actually didn't help, so it's commented out and almost the default parameters are used

In [26]:
from lightgbm import LGBMRegressor

In [27]:
# Grid Search
"""from sklearn.model_selection import GridSearchCV
param_grid = [{'min_data_in_leaf': [100, 200, 400], 
               'num_leaves': [5, 10, 20],
               'boosting_type': ['gbdt', 'dart'],
               'n_estimators': [250, 500, 1000],
               'learning_rate': [0.1, 0.05, 0.01]
              }]
param_grid2 = [{'n_estimators': [250, 500, 1000, 2000],
               'learning_rate': [0.1, 0.05, 0.01, 0.001]
              }]

# Get it ready
grid_search = GridSearchCV(forest, param_grid2, cv=7, verbose=1, n_jobs=-1)"""

"from sklearn.model_selection import GridSearchCV\nparam_grid = [{'min_data_in_leaf': [100, 200, 400], \n               'num_leaves': [5, 10, 20],\n               'boosting_type': ['gbdt', 'dart'],\n               'n_estimators': [250, 500, 1000],\n               'learning_rate': [0.1, 0.05, 0.01]\n              }]\nparam_grid2 = [{'n_estimators': [250, 500, 1000, 2000],\n               'learning_rate': [0.1, 0.05, 0.01, 0.001]\n              }]\n\n# Get it ready\ngrid_search = GridSearchCV(forest, param_grid2, cv=7, verbose=1, n_jobs=-1)"

In [28]:
#grid_search.fit(train_ready, y_train)

In [29]:
#grid_search.best_params_

In [30]:
#forest = grid_search.best_estimator_

In [31]:
forest = LGBMRegressor(n_estimators=500, learning_rate = 0.01)

In [32]:
forest.fit(train_ready, y_train)

LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
              importance_type='split', learning_rate=0.01, max_depth=-1,
              min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
              n_estimators=500, n_jobs=-1, num_leaves=31, objective=None,
              random_state=None, reg_alpha=0.0, reg_lambda=0.0, silent=True,
              subsample=1.0, subsample_for_bin=200000, subsample_freq=0)

In [33]:
y_pred = forest.predict(test_ready)

# Submit

In [34]:
# Read in sample csv
sample_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')

In [35]:
sample_df['SalePrice'] = y_pred

In [36]:
# Write to a new csv
sample_df.to_csv('predictions.csv', index=False) # Be sure to not include the index

In [37]:
# LightGBM (no grid search) - 0.13334
# LightGBM (grid search) - nothing better

# Possible Ways to Improve

In [38]:
# Ensemble learning to get better results
# Mess with the imputer
# Feature engineering with the slew of columns